In [16]:
import os
import json
import pandas as pd 
from pathlib import Path


base_path="/Users/gokstu/Documents/GitHub/pulse/data/"


In [ ]:
# 1.GETTING DATA FROM GIT

# 2.IMPORTING THE MODULES

# 3.MYSQL CONNECTION 
#     -CREATE TABLE IN db
#     -CONNECTED TO PYTHON 

# 4.EXTRACTING AND LOADING .JSON FILES 

# 5.CREATING THE DATA FRAME AND CLEANING THE DATA by dropna AND drop_duplicate

# 6.PUSHING THE DATA TO MYSQL.DB

In [70]:
import mysql.connector

mydb = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root1702",
    database="phonepe"
)
mycursor = mydb.cursor()
print("connection successful",mydb.is_connected())


def insertIntoTables(dataframe, table_name, columns):
    for index,row in dataframe.iterrows():
        col = ",".join(columns)
        placeholder = "%s,"*(len(columns)-1)+"%s"
        query = f"INSERT INTO {table_name} ({col}) VALUES ({placeholder})"
        values = []
        # values = tuple(row[col] for col in columns)
        for col in columns:
            values.append(row[col])
        mycursor.execute(query,tuple(values))
    mydb.commit()

connection successful True


In [ ]:
# extraction of datas to create a data frame. (USED FROM THE REFERENCE COLLAB)

# 1. AGGREGATED TRANSACTIONS


path = base_path + "aggregated/transaction/country/india/state/"
agg_state_list = os.listdir(path)
# print(agg_state_list)

clm={'state':[] ,'year':[], 'quarter':[], 'transaction_name':[], 'transaction_count':[], 'transaction_amount':[]}
for i in agg_state_list:
    p_i=path+i+"/"
    agg_yr=os.listdir(p_i)
    for j in agg_yr:
        p_j=p_i+j+"/"
        agg_yr_list=os.listdir(p_j)
        for k in agg_yr_list:
            p_k=p_j+k
            data=open(p_k,'r')
            d=json.load(data)
            for z in d['data']['transactionData']:
                name=z['name']
                count=z['paymentInstruments'][0]['count']
                amount=z['paymentInstruments'][0]['amount']
                clm['transaction_name'].append(name)
                clm['transaction_count'].append(count)
                clm['transaction_amount'].append(amount)
                clm['state'].append(i)
                clm['year'].append(j)
                clm['quarter'].append(int(k.strip('.json')))

aggregated_transactions=pd.DataFrame(clm).dropna().drop_duplicates()



insertIntoTables(aggregated_transactions, "aggregated_transactions", ["state", "year", "quarter", "transaction_name", "transaction_count", "transaction_amount"])

In [ ]:
# 2. AGGREGATED USERS


path =  base_path + "aggregated/user/country/india/state/"
agg_state_list = os.listdir(path)
# print(agg_state_list)


clm = {'state':[], 'year':[], 'quarter':[], 'user_brand':[], 'user_count':[], 'user_percentage':[]}

for i in agg_state_list:
    p_i = path + i + "/"        
    agg_yr = os.listdir(p_i)
    for j in agg_yr:
        p_j = p_i + j + "/"
        agg_yr_list = os.listdir(p_j)
        for k in agg_yr_list:
            p_k = p_j + k
            data=open(p_k, 'r')
            d = json.load(data)
            users = d['data'].get('usersByDevice')  
            if users is None:                        
                continue
            for z in users:                         
                clm['user_brand'].append(z['brand'])
                clm['user_count'].append(z['count'])
                clm['user_percentage'].append(z['percentage'])
                clm['state'].append(i)
                clm['year'].append(j)
                clm['quarter'].append(int(k.strip('.json')))


aggregated_users=pd.DataFrame(clm).dropna().drop_duplicates()

len(aggregated_users['user_brand'])

insertIntoTables(aggregated_users, "aggregated_users", ["state", "year", "quarter", "user_brand", "user_count", "user_percentage"])

['andaman-&-nicobar-islands', 'tamil-nadu', 'lakshadweep', 'telangana', 'manipur', 'haryana', 'gujarat', 'sikkim', 'delhi', 'west-bengal', 'uttar-pradesh', 'goa', 'punjab', 'arunachal-pradesh', 'karnataka', 'jammu-&-kashmir', 'maharashtra', 'odisha', 'madhya-pradesh', 'rajasthan', 'andhra-pradesh', 'chandigarh', 'kerala', 'chhattisgarh', 'tripura', 'mizoram', 'himachal-pradesh', 'dadra-&-nagar-haveli-&-daman-&-diu', 'ladakh', 'assam', 'meghalaya', 'uttarakhand', 'puducherry', 'bihar', 'jharkhand', 'nagaland']


In [ ]:
# # 3. AGGREGATED INSURANCE

path = base_path + "aggregated/insurance/country/india/state/"
agg_state_list = os.listdir(path)
# print(agg_state_list)

clm = {'state':[], 'year':[], 'quarter':[], 'insurance_name':[], 'insurance_count':[], 'insurance_amount':[]}


for i in agg_state_list:
    p_i = path + i + "/"
    agg_yr = os.listdir(p_i)
    for j in agg_yr:
        p_j = p_i + j + "/"
        agg_yr_list = os.listdir(p_j)
        for k in agg_yr_list:
            p_k = p_j + k
            data = open(p_k, 'r')
            d = json.load(data)
            insurance_data = d['data'].get('transactionData')
            if insurance_data is None:
                continue
            for z in insurance_data:
                name   = z['name']
                count  = z['paymentInstruments'][0]['count']
                amount = z['paymentInstruments'][0]['amount']
                clm['insurance_name'].append(name)
                clm['insurance_count'].append(count)
                clm['insurance_amount'].append(amount)
                clm['state'].append(i)
                clm['year'].append(j)
                clm['quarter'].append(int(k.strip('.json')))

aggregated_insurance = pd.DataFrame(clm).dropna().drop_duplicates()
# print(aggregated_insurance)
len(aggregated_insurance['insurance_name'])

insertIntoTables(aggregated_insurance, "aggregated_insurance", ["state", "year", "quarter", "insurance_name", "insurance_count", "insurance_amount"])

['andaman-&-nicobar-islands', 'tamil-nadu', 'lakshadweep', 'telangana', 'manipur', 'haryana', 'gujarat', 'sikkim', 'delhi', 'west-bengal', 'uttar-pradesh', 'goa', 'punjab', 'arunachal-pradesh', 'karnataka', 'jammu-&-kashmir', 'maharashtra', 'odisha', 'madhya-pradesh', 'rajasthan', 'andhra-pradesh', 'chandigarh', 'kerala', 'chhattisgarh', 'tripura', 'mizoram', 'himachal-pradesh', 'dadra-&-nagar-haveli-&-daman-&-diu', 'ladakh', 'assam', 'meghalaya', 'uttarakhand', 'puducherry', 'bihar', 'jharkhand', 'nagaland']


In [ ]:
# 4. MAP TRANSACTIONS 


map_path = base_path + "map/transaction/hover/country/india/state/"
map_state_list = os.listdir(map_path)
# print(map_state_list)

map_clm = {'state':[], 'year':[], 'quarter':[], 'district_name':[], 'transaction_count':[], 'transaction_amount':[]}

for i in map_state_list:
    p_i=map_path+i+"/"
    agg_yr=os.listdir(p_i)
    for j in agg_yr:
        p_j=p_i+j+"/"
        agg_yr_list=os.listdir(p_j)
        for k in agg_yr_list:
            p_k=p_j+k
            data=open(p_k,'r')
            d=json.load(data)
            map_data =  d['data']['hoverDataList']
            if map_data is None:
                continue
            for z in map_data:
                name=z['name']
                count=z['metric'][0]['count']
                amount=z['metric'][0]['amount']
                map_clm['district_name'].append(name)
                map_clm['transaction_count'].append(count)
                map_clm['transaction_amount'].append(amount)
                map_clm['state'].append(i)
                map_clm['year'].append(j)
                map_clm['quarter'].append(int(k.strip('.json')))

map_transactions=pd.DataFrame(map_clm).dropna().drop_duplicates()
# print(map_transactions)
len(map_transactions['district_name'])

insertIntoTables(map_transactions, "map_transactions", ["state", "year", "quarter", "district_name", "transaction_count", "transaction_amount"])






['andaman-&-nicobar-islands', 'tamil-nadu', 'lakshadweep', 'telangana', 'manipur', 'haryana', 'gujarat', 'sikkim', 'delhi', 'west-bengal', 'uttar-pradesh', 'goa', 'punjab', 'arunachal-pradesh', 'karnataka', 'jammu-&-kashmir', 'maharashtra', 'odisha', 'madhya-pradesh', 'rajasthan', 'andhra-pradesh', 'chandigarh', 'kerala', 'chhattisgarh', 'tripura', 'mizoram', 'himachal-pradesh', 'dadra-&-nagar-haveli-&-daman-&-diu', 'ladakh', 'assam', 'meghalaya', 'uttarakhand', 'puducherry', 'bihar', 'jharkhand', 'nagaland']


In [ ]:
# 5. MAP USERS 

map_path = base_path + "map/user/hover/country/india/state/"
map_state_list = os.listdir(map_path)
# print(map_state_list)

map_clm = {'state':[], 'year':[], 'quarter':[], 'user_district':[], 'user_registered':[], 'user_appopens':[]}

for i in map_state_list:
    p_i = map_path + i + "/"
    agg_yr = os.listdir(p_i)
    
    for j in agg_yr:
        p_j = p_i + j + "/"
        agg_yr_list= os.listdir(p_j)        
        for k in agg_yr_list:
            p_k = p_j + k
            data = open(p_k, 'r') 
            d = json.load(data)         
            map_data = d['data']['hoverData']
            if map_data is not None:
                   for district_name, values in map_data.items():
                    map_clm['user_district'].append(district_name)
                    map_clm['user_registered'].append(values['registeredUsers'])
                    map_clm['user_appopens'].append(values['appOpens'])
                    map_clm['state'].append(i)
                    map_clm['year'].append(j)
                    map_clm['quarter'].append(int(k.strip('.json')))

map_users = pd.DataFrame(map_clm).dropna().drop_duplicates()
# print(map_users)
len(map_users['user_registered'])

insertIntoTables(map_users, "map_users", ["state", "year", "quarter", "user_district", "user_registered", "user_appopens"])

['andaman-&-nicobar-islands', 'tamil-nadu', 'lakshadweep', 'telangana', 'manipur', 'haryana', 'gujarat', 'sikkim', 'delhi', 'west-bengal', 'uttar-pradesh', 'goa', 'punjab', 'arunachal-pradesh', 'karnataka', 'jammu-&-kashmir', 'maharashtra', 'odisha', 'madhya-pradesh', 'rajasthan', 'andhra-pradesh', 'chandigarh', 'kerala', 'chhattisgarh', 'tripura', 'mizoram', 'himachal-pradesh', 'dadra-&-nagar-haveli-&-daman-&-diu', 'ladakh', 'assam', 'meghalaya', 'uttarakhand', 'puducherry', 'bihar', 'jharkhand', 'nagaland']


In [ ]:
# 6. USER INSURANCE 

map_path = base_path + "map/insurance/hover/country/india/state/"
map_state_list = os.listdir(map_path)
# print(map_state_list)

map_clm = {'state':[], 'year':[], 'quarter':[], 'district_name':[], 'insurance_count':[], 'insurance_amount':[]}

for i in map_state_list:
    p_i=map_path+i+"/"
    agg_yr=os.listdir(p_i)
    for j in agg_yr:
        p_j=p_i+j+"/"
        agg_yr_list=os.listdir(p_j)
        for k in agg_yr_list:
            p_k=p_j+k
            data=open(p_k,'r')
            d=json.load(data)
            map_data =  d['data']['hoverDataList']
            if map_data is not None:
            
             for z in map_data:
                name=z['name']
                count=z['metric'][0]['count']
                amount=z['metric'][0]['amount']
                map_clm['district_name'].append(name)
                map_clm['insurance_count'].append(count)
                map_clm['insurance_amount'].append(amount)
                map_clm['state'].append(i)
                map_clm['year'].append(j)
                map_clm['quarter'].append(int(k.strip('.json')))

map_insurance = pd.DataFrame(map_clm).dropna().drop_duplicates()
# print(map_insurance)
len(map_insurance['district_name'])

insertIntoTables(map_insurance, "map_insurance", ["state", "year", "quarter", "district_name", "insurance_count", "insurance_amount"])



['andaman-&-nicobar-islands', 'tamil-nadu', 'lakshadweep', 'telangana', 'manipur', 'haryana', 'gujarat', 'sikkim', 'delhi', 'west-bengal', 'uttar-pradesh', 'goa', 'punjab', 'arunachal-pradesh', 'karnataka', 'jammu-&-kashmir', 'maharashtra', 'odisha', 'madhya-pradesh', 'rajasthan', 'andhra-pradesh', 'chandigarh', 'kerala', 'chhattisgarh', 'tripura', 'mizoram', 'himachal-pradesh', 'dadra-&-nagar-haveli-&-daman-&-diu', 'ladakh', 'assam', 'meghalaya', 'uttarakhand', 'puducherry', 'bihar', 'jharkhand', 'nagaland']


In [80]:
# 7. TOP TRANSACTIONS 

top_path = base_path + "top/transaction/country/india/state/"
top_state_list = os.listdir(top_path)
# print(top_state_list)

# top_clm_district = {'state':[], 'year':[], 'quarter':[], 'district':[], 'transaction_count':[], 'transaction_amount':[]}
top_clm_pincode = {'state':[], 'year':[], 'quarter':[], 'pincode':[], 'transaction_count':[], 'transaction_amount':[]}

for i in top_state_list:
    p_i=top_path+"/"+i+"/"
    agg_yr=os.listdir(p_i)
    for j in agg_yr:
        p_j=p_i+j+"/"
        agg_yr_list=os.listdir(p_j)
        for k in agg_yr_list:
            p_k=p_j+k
            data=open(p_k,'r')
            d=json.load(data)
            top_data = d['data'].get('pincodes')
            if top_data is not None: 
                for z in top_data:
                    pincode=z['entityName']
                    count=z['metric']['count']
                    amount=z['metric']['amount']
                    top_clm_pincode['pincode'].append(pincode)
                    top_clm_pincode['transaction_count'].append(count)
                    top_clm_pincode['transaction_amount'].append(amount)
                    top_clm_pincode['state'].append(i)
                    top_clm_pincode['year'].append(j)
                    top_clm_pincode['quarter'].append(int(k.strip('.json')))
            

top_transactions_pincode=pd.DataFrame(top_clm_pincode).dropna().drop_duplicates()
# print(top_transactions_pincode)
len(top_transactions_pincode['pincode'])

insertIntoTables(top_transactions_pincode, "top_transactions", ["state", "year", "quarter", "pincode", "transaction_count", "transaction_amount"])

In [ ]:
# 8. TOP USERS 

top_path = base_path + "top/user/country/india/state/"
top_state_list = os.listdir(top_path)
# print(top_state_list)

top_user_clm = {'state':[], 'year':[], 'quarter':[], 'pincode':[], 'registered_users':[]}

for i in top_state_list:
    p_i=top_path+"/"+i+"/"
    agg_yr=os.listdir(p_i)
    for j in agg_yr:
        p_j=p_i+j+"/"
        agg_yr_list=os.listdir(p_j)
        for k in agg_yr_list:
            p_k=p_j+k
            data=open(p_k,'r')
            d=json.load(data)
            top_data = d['data'].get('pincodes')
            if top_data is  None:
                continue
            for z in top_data:
                 pincode=z['name']
                 registered_users=z['registeredUsers']
                 top_user_clm['pincode'].append(pincode)
                 top_user_clm['registered_users'].append(registered_users)
                 top_user_clm['state'].append(i)
                 top_user_clm['year'].append(j)
                 top_user_clm['quarter'].append(int(k.strip('.json')))

top_users=pd.DataFrame(top_user_clm).dropna().drop_duplicates()
# print(top_users)
len(top_users['registered_users'])

insertIntoTables(top_users, "top_users", ["state", "year", "quarter", "pincode", "registered_users"])

['andaman-&-nicobar-islands', 'tamil-nadu', 'lakshadweep', 'telangana', 'manipur', 'haryana', 'gujarat', 'sikkim', 'delhi', 'west-bengal', 'uttar-pradesh', 'goa', 'punjab', 'arunachal-pradesh', 'karnataka', 'jammu-&-kashmir', 'maharashtra', 'odisha', 'madhya-pradesh', 'rajasthan', 'andhra-pradesh', 'chandigarh', 'kerala', 'chhattisgarh', 'tripura', 'mizoram', 'himachal-pradesh', 'dadra-&-nagar-haveli-&-daman-&-diu', 'ladakh', 'assam', 'meghalaya', 'uttarakhand', 'puducherry', 'bihar', 'jharkhand', 'nagaland']


In [ ]:
# 9. TOP INSURANCE 

top_path = base_path + "top/insurance/country/india/state/"
top_state_list = os.listdir(top_path)
# print(top_state_list)

top_insurance_clm = {'state':[], 'year':[], 'quarter':[], 'pincode':[], 'insurance_count':[], 'insurance_amount':[]}

for i in top_state_list:
    p_i = top_path + "/" + i + "/"
    agg_yr = os.listdir(p_i)
    for j in agg_yr:
        p_j = p_i + j + "/"
        agg_yr_list = os.listdir(p_j)
        for k in agg_yr_list:
            p_k = p_j + k
            data = open(p_k, 'r')
            d = json.load(data)
            top_data = d['data'].get('pincodes')
            if top_data is None:
                continue
            for z in top_data:
                pincode   = z['entityName']
                count  = z['metric']['count']
                amount = z['metric']['amount']
                top_insurance_clm['pincode'].append(pincode)
                top_insurance_clm['insurance_count'].append(count)
                top_insurance_clm['insurance_amount'].append(amount)
                top_insurance_clm['state'].append(i)
                top_insurance_clm['year'].append(j)
                top_insurance_clm['quarter'].append(int(k.strip('.json')))


top_insurance = pd.DataFrame(top_insurance_clm).dropna().drop_duplicates()
# print(top_insurance)
len(top_insurance['year'])

insertIntoTables(top_insurance, "top_insurance", ["state", "year", "quarter", "pincode", "insurance_count", "insurance_amount"])

['andaman-&-nicobar-islands', 'tamil-nadu', 'lakshadweep', 'telangana', 'manipur', 'haryana', 'gujarat', 'sikkim', 'delhi', 'west-bengal', 'uttar-pradesh', 'goa', 'punjab', 'arunachal-pradesh', 'karnataka', 'jammu-&-kashmir', 'maharashtra', 'odisha', 'madhya-pradesh', 'rajasthan', 'andhra-pradesh', 'chandigarh', 'kerala', 'chhattisgarh', 'tripura', 'mizoram', 'himachal-pradesh', 'dadra-&-nagar-haveli-&-daman-&-diu', 'ladakh', 'assam', 'meghalaya', 'uttarakhand', 'puducherry', 'bihar', 'jharkhand', 'nagaland']


In [ ]:


# print((aggregated_transactions))
# for index,row in aggregated_transactions.iterrows():
#     state = row['state']
#     year = row['year']
#     quarter = row['quarter']
#     transaction_name = row['transaction_name']
#     transaction_count = row['transaction_count']
#     transaction_amount = row['transaction_amount']
#     query = "INSERT INTO aggregated_transactions (state, year, quarter, transaction_name, transaction_count, transaction_amount) VALUES (%s, %s, %s, %s, %s, %s)"
#     mycursor.execute(query,(state, year, quarter, transaction_name, transaction_count, transaction_amount))
# mydb.commit()
